In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader
import pandas as pd

# 하이퍼파라미터 설정
num_epochs = 300
batch_size = 32
learning_rate = 0.001
dropout_rate = 0.3

# 데이터 전처리 및 증강
transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 데이터셋 로드 (폴더에서 직접 불러오기)
data_dir = r"C:\Users\user\OneDrive\Desktop\Resnet18\Resnet18\data\last_camter_tent_dataset"
train_dir = os.path.join(data_dir, 'train')
valid_dir = os.path.join(data_dir, 'val')

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
valid_dataset = datasets.ImageFolder(root=valid_dir, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(dataset=valid_dataset, batch_size=batch_size, shuffle=False)

# 모델 초기화
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features

# 드롭아웃 레이어 추가
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, len(train_dataset.classes)),  # 클래스 수에 맞게 출력층 수정
    nn.Dropout(dropout_rate)  # 드롭아웃 추가
)

# 손실 함수 및 최적화 알고리즘 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# GPU 사용 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 학습 및 검증 함수
def train_and_validate(model, train_loader, valid_loader, criterion, optimizer, num_epochs):
    best_valid_accuracy = 0.0
    best_epoch = 0
    best_train_loss = 0.0
    best_train_accuracy = 0.0
    best_valid_loss = 0.0

    train_losses = []
    valid_losses = []
    train_accuracies = []
    valid_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        total_train = 0

        # 학습
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_accuracy = train_correct / total_train
        train_loss /= len(train_loader)

        # 검증
        model.eval()
        valid_loss = 0.0
        valid_correct = 0
        total_valid = 0

        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                valid_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_valid += labels.size(0)
                valid_correct += (predicted == labels).sum().item()

        valid_accuracy = valid_correct / total_valid
        valid_loss /= len(valid_loader)

        # 결과 출력
        print(f'Epoch [{epoch+1}/{num_epochs}], '
            f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, '
            f'Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_accuracy:.4f}')
        
        # 손실 및 정확도 저장
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        train_accuracies.append(train_accuracy)
        valid_accuracies.append(valid_accuracy)

        # 최고 검증 정확도 모델 저장
        if valid_accuracy > best_valid_accuracy:
            best_valid_accuracy = valid_accuracy
            best_epoch = epoch + 1
            best_train_loss = train_loss
            best_train_accuracy = train_accuracy
            best_valid_loss = valid_loss

            torch.save(model.state_dict(), 'best_model.pth')

    # 학습 종료 후 가장 높은 검증 정확도의 결과 출력
    print("\n🔥 학습 종료! 가장 높은 검증 정확도를 기록한 에포크 🔥")
    print(f"✅ Best Epoch: {best_epoch}")
    print(f"📉 Best Train Loss: {best_train_loss:.4f}")
    print(f"🎯 Best Train Accuracy: {best_train_accuracy:.4f}")
    print(f"📉 Best Valid Loss: {best_valid_loss:.4f}")
    print(f"🎯 Best Valid Accuracy: {best_valid_accuracy:.4f}")

    # 결과를 DataFrame으로 변환 후 CSV 파일로 저장
    results_df = pd.DataFrame({
        'Epoch': range(1, len(train_losses) + 1),
        'Train Loss': train_losses,
        'Valid Loss': valid_losses,
        'Train Accuracy': train_accuracies,
        'Valid Accuracy': valid_accuracies
    })
    results_df.to_csv('resnet18_epoch1000_results.csv', index=False)

# 모델 학습
train_and_validate(model, train_loader, valid_loader, criterion, optimizer, num_epochs)


Epoch [1/300], Train Loss: 2.0646, Train Accuracy: 0.3123, Valid Loss: 90.7643, Valid Accuracy: 0.1809
Epoch [2/300], Train Loss: 1.7852, Train Accuracy: 0.3507, Valid Loss: 15.6858, Valid Accuracy: 0.2234
Epoch [3/300], Train Loss: 1.5921, Train Accuracy: 0.4192, Valid Loss: 1.6680, Valid Accuracy: 0.4362
Epoch [4/300], Train Loss: 1.4969, Train Accuracy: 0.4274, Valid Loss: 1.7008, Valid Accuracy: 0.4043
Epoch [5/300], Train Loss: 1.3544, Train Accuracy: 0.4219, Valid Loss: 1.4220, Valid Accuracy: 0.4681
Epoch [6/300], Train Loss: 1.2205, Train Accuracy: 0.5178, Valid Loss: 1.4916, Valid Accuracy: 0.5000
Epoch [7/300], Train Loss: 1.2823, Train Accuracy: 0.4904, Valid Loss: 1.3664, Valid Accuracy: 0.4787
Epoch [8/300], Train Loss: 1.3008, Train Accuracy: 0.4712, Valid Loss: 1.9011, Valid Accuracy: 0.5319
Epoch [9/300], Train Loss: 1.3589, Train Accuracy: 0.4767, Valid Loss: 1.3927, Valid Accuracy: 0.4894
Epoch [10/300], Train Loss: 1.2651, Train Accuracy: 0.4904, Valid Loss: 1.4476, 

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader

# ====== GPU 설정 ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 사용 중인 디바이스: {device}")

# ====== 경로 설정 (수정된 폴더 구조 반영) ======
base_dir = r"C:\Users\user\OneDrive\Desktop\Resnet18\Resnet18\data\last_camter_tent_dataset"
model_path = r"C:\Users\user\OneDrive\Desktop\Resnet18\Resnet18\model\best_model.pth"
output_path = r"C:\Users\user\OneDrive\Desktop\Resnet18\Resnet18\csv\evaluation_results.txt"  # ✅ 결과 저장 경로 추가

# ====== 데이터 전처리 (학습 시 사용한 전처리와 동일) ======
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 테스트에서는 RandomCrop 대신 Resize
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ====== 전체 데이터셋 불러오기 (train, val, test 모두 포함) ======
datasets_dict = {}
data_loaders = {}
accuracy_dict = {}

for split in ["train", "val", "test"]:
    dataset_path = os.path.join(base_dir, split)
    
    if os.path.exists(dataset_path):
        datasets_dict[split] = datasets.ImageFolder(root=dataset_path, transform=transform)
        data_loaders[split] = DataLoader(datasets_dict[split], batch_size=1, shuffle=False)
        print(f"✅ {split} 데이터셋 로드 완료: {len(datasets_dict[split])}개 이미지")
    else:
        print(f"🚨 {split} 폴더를 찾을 수 없습니다: {dataset_path}")

# 클래스 이름 가져오기
class_names = datasets_dict["train"].classes if "train" in datasets_dict else datasets_dict["test"].classes

# ====== 학습된 모델 로드 ======
model = models.resnet18(pretrained=False)  # ResNet18 모델 생성
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, len(class_names))
)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# ====== 전체 데이터셋 예측 수행 ======
correct_total = 0
total_images = 0

results = []  # ✅ 결과 저장 리스트 추가
results.append("📢 전체 데이터셋 테스트 결과:\n")
print("\n📢 전체 데이터셋 테스트 결과:")

with torch.no_grad():
    for split, loader in data_loaders.items():
        correct = 0
        total = 0
        
        results.append(f"\n🔹 {split.upper()} 데이터셋 테스트 시작...\n")
        print(f"\n🔹 {split.upper()} 데이터셋 테스트 시작...")

        for idx, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            pred_label = class_names[predicted.item()]
            true_label = class_names[labels.item()]

            total += 1
            total_images += 1
            if predicted == labels:
                correct += 1
                correct_total += 1
                results.append(f"[✅ 정답] 이미지 {idx+1}: 예측={pred_label}, 실제={true_label}\n")
                print(f"[✅ 정답] 이미지 {idx+1}: 예측={pred_label}, 실제={true_label}")
            else:
                results.append(f"[❌ 오답] 이미지 {idx+1}: 예측={pred_label}, 실제={true_label}\n")
                print(f"[❌ 오답] 이미지 {idx+1}: 예측={pred_label}, 실제={true_label}")

        # 데이터셋별 정확도 출력
        accuracy = (correct / total) * 100 if total > 0 else 0
        accuracy_dict[split] = (correct, total, accuracy)
        results.append(f"🔥 {split.upper()} 데이터셋 정확도: {accuracy:.2f}% ({correct}/{total} 맞춤)\n")
        print(f"🔥 {split.upper()} 데이터셋 정확도: {accuracy:.2f}% ({correct}/{total} 맞춤)")

# ====== 전체 데이터셋 최종 정확도 출력 ======
final_accuracy = (correct_total / total_images) * 100 if total_images > 0 else 0
results.append(f"\n🎯 전체 데이터셋 최종 정확도: {final_accuracy:.2f}% ({correct_total}/{total_images} 맞춤)\n")
print(f"\n🎯 전체 데이터셋 최종 정확도: {final_accuracy:.2f}% ({correct_total}/{total_images} 맞춤)")

# ====== 최종 요약 출력 ======
results.append("\n📊 전체 평가 결과 요약:\n")
print("\n📊 전체 평가 결과 요약:")
for split, (correct, total, accuracy) in accuracy_dict.items():
    results.append(f"🔹 {split.upper()} 데이터셋: 정확도 {accuracy:.2f}% ({correct}/{total} 맞춤)\n")
    print(f"🔹 {split.upper()} 데이터셋: 정확도 {accuracy:.2f}% ({correct}/{total} 맞춤)")

# ✅ 결과를 txt 파일로 저장
with open(output_path, "w", encoding="utf-8") as f:
    f.writelines(results)

print(f"\n📂 결과가 {output_path} 파일에 저장되었습니다.")


✅ 사용 중인 디바이스: cuda
✅ train 데이터셋 로드 완료: 365개 이미지
✅ val 데이터셋 로드 완료: 94개 이미지
✅ test 데이터셋 로드 완료: 24개 이미지


C:\Users\user\AppData\Local\Temp\ipykernel_23468\1751390804.py:49: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=de


📢 전체 데이터셋 테스트 결과:

🔹 TRAIN 데이터셋 테스트 시작...
[✅ 정답] 이미지 1: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 2: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 3: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 4: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 5: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[❌ 오답] 이미지 6: 예측=1 헬리녹스 노나돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 7: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 8: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 9: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[❌ 오답] 이미지 10: 예측=1 헬리녹스 노나돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 11: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 12: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[❌ 오답] 이미지 13: 예측=1 헬리녹스 노나돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 14: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 15: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 16: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 17: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 18: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 19: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 20: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 정답] 이미지 21: 예측=0 헬리녹스 알파인돔, 실제=0 헬리녹스 알파인돔
[✅ 